# 01 — Problema e benchmark

## Objetivo

Qual referência simples deve orientar as próximas etapas?

Comparamos Regressão Logística e Random Forest na mesma validação. O teste fica
reservado até o notebook final.

In [1]:
from pathlib import Path
import sys

ponto_atual = Path.cwd().resolve()
RAIZ = next(
    caminho for caminho in (ponto_atual, *ponto_atual.parents)
    if (caminho / "data" / "raw" / "UCI_Credit_Card.csv").exists()
)
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))


import time
import pandas as pd

from src.auxiliares import (
    avaliar_probabilidades,
    carregar_base_preparada,
    criar_modelo_floresta,
    criar_modelo_logistico,
    separar_dados,
)
from src.visual_utils import grafico_comparacao_modelos

## Como separar features, target e conjuntos?

In [2]:
dados = carregar_base_preparada(RAIZ)
X_treino, X_validacao, X_teste, y_treino, y_validacao, y_teste = separar_dados(dados)

pd.DataFrame({
    "conjunto": ["treino", "validação", "teste protegido"],
    "registros": [len(X_treino), len(X_validacao), len(X_teste)],
    "proporcao_positiva": [y_treino.mean(), y_validacao.mean(), "protegido"],
})

,conjunto,registros,proporcao_positiva
0,treino,18000,0.221222
1,validação,6000,0.221167
2,teste protegido,6000,protegido


O split estratificado produz 60% para treino, 20% para validação e 20% para
teste. ID e target não entram nas 23 features.

## O que a Regressão Logística entrega?

In [3]:
modelo_logistico = criar_modelo_logistico()
inicio = time.perf_counter()
modelo_logistico.fit(X_treino, y_treino)
tempo_logistico = time.perf_counter() - inicio
previsoes_logisticas = modelo_logistico.predict(X_validacao)
probabilidades_logisticas = modelo_logistico.predict_proba(X_validacao)[:, 1]

In [4]:
pd.DataFrame({
    "classe_prevista": previsoes_logisticas[:5],
    "probabilidade": probabilidades_logisticas[:5],
})

,classe_prevista,probabilidade
0,0,0.259325
1,0,0.144900
2,0,0.056384
3,0,0.111899
4,0,0.105457


## O bagging melhora a referência?

In [5]:
modelo_floresta = criar_modelo_floresta()
inicio = time.perf_counter()
modelo_floresta.fit(X_treino, y_treino)
tempo_floresta = time.perf_counter() - inicio
previsoes_floresta = modelo_floresta.predict(X_validacao)
probabilidades_floresta = modelo_floresta.predict_proba(X_validacao)[:, 1]

## Como comparar os dois modelos?

In [6]:
resultados = pd.DataFrame([
    avaliar_probabilidades(
        "Regressão Logística", y_validacao, probabilidades_logisticas,
        tempo_treino=tempo_logistico,
    ),
    avaliar_probabilidades(
        "Random Forest", y_validacao, probabilidades_floresta,
        tempo_treino=tempo_floresta,
    ),
])
resultados

,modelo,limiar,precision,recall,f1,pr_auc,vn,fp,fn,vp,tempo_treino_s
0,Regressão Logística,0.5,0.716129,0.250942,0.371652,0.500144,4541,132,994,333,0.177939
1,Random Forest,0.5,0.654667,0.370008,0.472797,0.529364,4414,259,836,491,1.506843


In [7]:
fig = grafico_comparacao_modelos(resultados)
fig.show()

## Resultado

A Random Forest melhora o ranking probabilístico em relação à logística e será
o exemplo prático de bagging. Precision, Recall e F1 ainda usam limiar 0,50.